In [1]:
import numpy as np
import pandas as pd
import dagshub 
import mlflow

c:\Users\win11\OneDrive\Desktop\MLops\Project\yt_plugin\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dagshub.init(repo_name = 'yt_chrome_plugin', repo_owner = 'AmanGupta8423', mlflow = True)
mlflow.set_experiment("Experiment2-max_features-tuning")
mlflow.set_tracking_uri('https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow')

Accessing as AmanGupta8423

Initialized MLflow to track repo "AmanGupta8423/yt_chrome_plugin"

Repository AmanGupta8423/yt_chrome_plugin initialized!

2026/11/26 17:38:10 INFO mlflow.tracking.fluent: Experiment with name 'Experiment2-max_features-tuning' does not exist. Creating a new experiment.


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.model_selection import train_test_split
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [4]:
df = pd.read_csv('./data/raw/dataset.csv').dropna(subset='clean_comment')
df.shape

(36662, 2)

In [5]:
def run_experiment_tfidf_max_features(max_features):
    vectorizer = TfidfVectorizer(max_features= max_features, ngram_range=(1,3))

    X_train,X_test,y_train,y_test = train_test_split(df['clean_comment'], df['category'], test_size = 0.2 , random_state = 42)

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    with mlflow.start_run() as run:
        mlflow.set_tag('mlflow.runName', f'Tfidf_trigram_max_features:{max_features}_RandomForest')
        mlflow.set_tag('experiment_type','feature_engineering')
        mlflow.set_tag('model_type', 'RandomForestClassifier')
        mlflow.set_tag('description','Max features for tfidf trigram')

        mlflow.log_param('vectorizer_name', 'TF-IDF')
        mlflow.log_param('ngrams_range','Trigram')   
        mlflow.log_param('max_features',max_features)  

        n_estimators = 100
        max_depth = 10
        mlflow.log_param('n_estimators',n_estimators)
        mlflow.log_param('max_depth',max_depth)

        model = RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth, random_state= 42)

        model.fit(X_train,y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test,y_pred)

        classification_rep = classification_report(y_test,y_pred,output_dict=True)
            
        for label,metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f'{label}_{metric}',value)

        conf_matrix = confusion_matrix(y_test,y_pred)
        plt.figure(figsize=(8,6))
        sns.heatmap(conf_matrix,annot = True, fmt = 'd',cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title(f'Confusion Matrix: RandomForest with Tfidf Trigram and max_features: {max_features}')
    
        os.makedirs("./reports", exist_ok=True)

        plt.savefig('./reports/confusion_matrix.png')
        mlflow.log_artifact('./reports/confusion_matrix.png')
        plt.close()

        mlflow.sklearn.log_model(model, name="random_forest_model",skops_trusted_types=["sklearn.tree._tree.Tree"])
        print(accuracy)


max_features = [1000,2000,3000,4000,5000,6000,7000,8000,9000,10000]

for max_feature in max_features:
    run_experiment_tfidf_max_features(max_feature)

0.6368471294149735
🏃 View run Tfidf_trigram_max_features:1000_RandomForest at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/3/runs/95aeb11305ea4ab0b4d91a80f156a825
🧪 View experiment at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/3
0.6420291831446884
🏃 View run Tfidf_trigram_max_features:2000_RandomForest at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/3/runs/2d3dd4844745497fb5bead9ba5688102
🧪 View experiment at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/3
0.6342561025501159
🏃 View run Tfidf_trigram_max_features:3000_RandomForest at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/3/runs/82f49a0ef4bd4c6b93a013c63449f4c1
🧪 View experiment at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/3
0.633437883540161
🏃 View run Tfidf_trigram_max_features:4000_RandomForest at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/e